# Эксперименты

Каждый эксперимент: **Гипотеза → Как проверялось → Результат**

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from src.preprocessing.clean import clean_all
from src.preprocessing.features import build_features
from src.models.evaluate import split_data, regression_metrics, print_metrics
from src.models.train import build_feature_matrix

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid')

In [ ]:
df = build_features(clean_all())
X, y, cat_indices, col_names = build_feature_matrix(df)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, random_state=RANDOM_SEED)
print(f'Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}')
print(f'Features: {col_names}')

# Числовые колонки для sklearn-пайплайнов (без категориальных)
num_cols = [c for i, c in enumerate(col_names) if i not in cat_indices]
X_train_num = X_train[num_cols].astype(float)
X_val_num = X_val[num_cols].astype(float)
X_test_num = X_test[num_cols].astype(float)

results = []

## Эксперимент 1 - RandomForest

**Гипотеза:** ансамблевый метод без настройки превзойдёт Ridge baseline за счёт нелинейных зависимостей.

**Как проверялось:** обучение на train, оценка на val по MAPE.

In [ ]:
rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)),
])
rf.fit(X_train_num, y_train)
m = regression_metrics(y_val, rf.predict(X_val_num))
print_metrics('RandomForest', m)
results.append({'model': 'RandomForest', **m})

## Эксперимент 2 - XGBoost

**Гипотеза:** градиентный бустинг лучше справится с выбросами и пропущенными значениями.

**Как проверялось:** early stopping на val, метрика RMSE.

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.05, random_state=RANDOM_SEED,
    early_stopping_rounds=50, n_jobs=-1, verbosity=0,
)
xgb_model.fit(
    X_train_num.fillna(-999), y_train,
    eval_set=[(X_val_num.fillna(-999), y_val)],
    verbose=False,
)
m = regression_metrics(y_val, xgb_model.predict(X_val_num.fillna(-999)))
print_metrics('XGBoost', m)
results.append({'model': 'XGBoost', **m})

## Эксперимент 3 - LightGBM (числовые признаки)

**Гипотеза:** LightGBM быстрее XGBoost и показывает схожее качество.

In [ ]:
lgb_num = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, random_state=RANDOM_SEED, n_jobs=-1,
)
lgb_num.fit(
    X_train_num, y_train,
    eval_set=[(X_val_num, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)
m = regression_metrics(y_val, lgb_num.predict(X_val_num))
print_metrics('LightGBM (num only)', m)
results.append({'model': 'LightGBM (num only)', **m})

## Эксперимент 4 - LightGBM + категориальные признаки

**Гипотеза:** добавление бренда, ОС и типа хранилища улучшит качество.

In [ ]:
lgb_full = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, random_state=RANDOM_SEED, n_jobs=-1,
)
lgb_full.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)
m = regression_metrics(y_val, lgb_full.predict(X_val))
print_metrics('LightGBM (all features)', m)
results.append({'model': 'LightGBM (all features)', **m})

## Эксперимент 5 - CatBoost

**Гипотеза:** CatBoost нативно обрабатывает категориальные признаки лучше LightGBM.

In [ ]:
cat_col_names = [col_names[i] for i in cat_indices]
X_train_cb = X_train.copy()
X_val_cb = X_val.copy()
X_test_cb = X_test.copy()
for col in cat_col_names:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_val_cb[col] = X_val_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

cb_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.05, random_seed=RANDOM_SEED,
    cat_features=cat_col_names, verbose=0, early_stopping_rounds=50,
)
cb_model.fit(X_train_cb, y_train, eval_set=(X_val_cb, y_val))
m = regression_metrics(y_val, cb_model.predict(X_val_cb))
print_metrics('CatBoost', m)
results.append({'model': 'CatBoost', **m})

## Эксперимент 6 - PCA + Ridge (уменьшение размерности)

**Гипотеза:** числовые признаки имеют коллинеарность; PCA может улучшить линейную модель.

In [ ]:
pca_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95, random_state=RANDOM_SEED)),
    ('model', Ridge()),
])
pca_pipe.fit(X_train_num, y_train)
n_comp = pca_pipe.named_steps['pca'].n_components_
print(f'PCA оставил {n_comp} компонент из {X_train_num.shape[1]}')

evr = pca_pipe.named_steps['pca'].explained_variance_ratio_
plt.figure(figsize=(8, 3))
plt.plot(np.cumsum(evr), marker='o', ms=4)
plt.axhline(0.95, color='red', ls='--', label='95%')
plt.xlabel('Количество компонент')
plt.ylabel('Объяснённая дисперсия')
plt.title('PCA: накопленная объяснённая дисперсия')
plt.legend()
plt.tight_layout()

m = regression_metrics(y_val, pca_pipe.predict(X_val_num))
print_metrics('Ridge + PCA', m)
results.append({'model': 'Ridge + PCA', **m})

## Эксперимент 7 - Стекинг

**Гипотеза:** комбинация RF + LightGBM через мета-модель Ridge даст лучший результат.

In [ ]:
stacking = StackingRegressor(
    estimators=[
        ('rf', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('model', RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)),
        ])),
        ('lgb', lgb.LGBMRegressor(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1)),
    ],
    final_estimator=Ridge(),
    cv=5, n_jobs=-1,
)
stacking.fit(X_train_num, y_train)
m = regression_metrics(y_val, stacking.predict(X_val_num))
print_metrics('Stacking (RF + LGB → Ridge)', m)
results.append({'model': 'Stacking', **m})

## Сводная таблица

In [ ]:
results_df = pd.DataFrame(results).sort_values('mape').reset_index(drop=True)
display(results_df)
results_df.to_csv(ROOT / 'data/processed/experiment_results.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(results_df['model'], results_df['mape'])
ax.set_xlabel('MAPE (%)')
ax.set_title('Сравнение моделей по MAPE (меньше = лучше)')
ax.invert_yaxis()
plt.tight_layout()

## Feature importance лучшей модели

In [ ]:
fi = pd.Series(lgb_full.feature_importances_, index=col_names).sort_values(ascending=False)
fi.head(15).plot(kind='bar', figsize=(10, 4))
plt.title('LightGBM feature importance (gain)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

## Финальная оценка на тесте

In [ ]:
best_name = results_df.iloc[0]['model']
print(f'Лучшая модель по val MAPE: {best_name}')

# Финальная оценка на тесте (только один раз!)
test_m = regression_metrics(y_test, lgb_full.predict(X_test))
print_metrics(f'LightGBM (all features) - TEST', test_m)